In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tensorflow.keras import Input
import seaborn as sns
from xgboost import XGBRegressor
from prophet import Prophet
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_parquet('/content/all_features_with_macro.parquet')

In [ ]:
df.columns

Index(['RegionID', 'RegionName', 'StateName', 'Date', 'HomeValue',
       'IncomeNeeded', 'Inventory', 'DaysToPending', 'RentValue',
       'RenterIncomeNeeded', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
       'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
       'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
       'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos',
       'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
       'HomeValue_Change', 'IncomeNeeded_Change', 'CPI_Change',
       'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
       'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
       'unrate_roll3'],
      dtype='object')

In [ ]:
mortgage_df = pd.read_parquet('/content/mortgage_long.parquet')

In [ ]:
mortgage_df

,RegionID,RegionName,StateName,Date,MortgageValue
0,394913,"New York, NY",NY,2012-01-31,1442.345690
1,753899,"Los Angeles, CA",CA,2012-01-31,1402.210418
2,394463,"Chicago, IL",IL,2012-01-31,653.935606
3,394514,"Dallas, TX",TX,2012-01-31,536.526162
4,394692,"Houston, TX",TX,2012-01-31,538.331663
...,...,...,...,...,...
64180,394787,"Lewiston, ID",ID,2025-09-30,1826.996877
64181,394570,"Enid, OK",OK,2025-09-30,690.300598
64182,395199,"Walla Walla, WA",WA,2025-09-30,2041.846102
64183,394444,"Carson City, NV",NV,2025-09-30,2404.687566


In [ ]:
# Function to normalize any date to the 1st day of its month
def normalize_to_month_start(df, date_col):
    """Converts a datetime column to the first day of the respective month."""
    # .dt.to_period('M') extracts the Month/Year period,
    # and .dt.to_timestamp() converts it back to the start of the month's timestamp
    return df[date_col].dt.to_period('M').dt.to_timestamp()

In [ ]:
mortgage_df["Date"] = normalize_to_month_start(mortgage_df, 'Date')
mortgage_df

,RegionID,RegionName,StateName,Date,MortgageValue
0,394913,"New York, NY",NY,2012-01-01,1442.345690
1,753899,"Los Angeles, CA",CA,2012-01-01,1402.210418
2,394463,"Chicago, IL",IL,2012-01-01,653.935606
3,394514,"Dallas, TX",TX,2012-01-01,536.526162
4,394692,"Houston, TX",TX,2012-01-01,538.331663
...,...,...,...,...,...
64180,394787,"Lewiston, ID",ID,2025-09-01,1826.996877
64181,394570,"Enid, OK",OK,2025-09-01,690.300598
64182,395199,"Walla Walla, WA",WA,2025-09-01,2041.846102
64183,394444,"Carson City, NV",NV,2025-09-01,2404.687566


In [ ]:
merged = pd.merge(
  df, mortgage_df,
  on=["RegionID","RegionName","StateName","Date"],
  how="inner"
)

In [ ]:
merged

,RegionID,RegionName,StateName,Date,HomeValue,IncomeNeeded,Inventory,DaysToPending,RentValue,RenterIncomeNeeded,...,CPI_Change,MortgageRate_Change,Unemployment_Change,cpi_lag1,cpi_roll3,mortgage_rate_lag1,mortgage_rate_roll3,unrate_lag1,unrate_roll3,MortgageValue
0,394304,"Akron, OH",OH,2018-05-01,141498.445820,37039.911190,3632.0,58.0,842.862503,33714.500133,...,0.000000,0.000000,0.000000,0.000,0.000000,0.0000,0.000000,0.0,0.000000,578.230936
1,394304,"Akron, OH",OH,2018-06-01,141997.352756,37126.302161,3806.0,55.0,844.909823,33796.392940,...,0.000901,-0.003489,0.052632,250.792,0.000000,4.5860,0.000000,3.8,0.000000,579.186971
2,394304,"Akron, OH",OH,2018-07-01,142509.730768,37144.015612,3917.0,54.0,844.722835,33788.913391,...,0.000781,-0.009300,-0.050000,251.018,251.008000,4.5700,4.561167,4.0,3.866667,578.395359
3,394304,"Akron, OH",OH,2018-08-01,143107.271515,37358.420177,3968.0,53.0,846.884637,33875.385472,...,0.001787,0.004970,0.000000,251.214,251.298333,4.5275,4.549167,3.8,3.866667,582.351584
4,394304,"Akron, OH",OH,2018-09-01,143845.022393,37757.841397,3918.0,56.0,850.004092,34000.163688,...,0.002062,0.017033,-0.026316,251.663,251.686333,4.5500,4.568333,3.8,3.766667,590.669795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999,395238,"Worcester, MA",MA,2025-03-01,471121.892980,128746.065420,1315.0,33.0,2118.639336,84745.573448,...,-0.000500,-0.028133,0.024390,319.775,319.492000,6.8425,6.816833,4.1,4.100000,2406.662822
8000,395238,"Worcester, MA",MA,2025-04-01,471093.632963,129493.319360,1495.0,26.0,2125.944172,85037.766887,...,0.002209,0.011278,0.000000,319.615,319.903667,6.6500,6.739167,4.2,4.166667,2425.157969
8001,395238,"Worcester, MA",MA,2025-05-01,470533.524469,130264.921899,1799.0,21.0,2130.885088,85235.403528,...,0.000809,0.013532,0.000000,320.321,320.172000,6.7250,6.730333,4.2,4.200000,2444.942673
8002,395238,"Worcester, MA",MA,2025-06-01,469919.352005,130117.860196,2037.0,20.0,2140.907753,85636.310100,...,0.002870,0.000220,-0.023810,320.580,320.800333,6.8160,6.786167,4.2,4.166667,2442.125250


In [ ]:
common_features = [
    "HomeValue", "Inventory", "DaysToPending", "MarketHeatIndex",
    "SalesCount", "NewConstruction",
    "cpi", "unrate", "mortgage_rate", "Month_Sin", "Month_Cos",
    "MortgageBurden", "Inventory_to_Sales", "CPI_Change",
    "MortgageRate_Change", "Unemployment_Change", "Rent_to_Mortgage_Ratio", "Sales_to_Inventory"
]
rent_features = common_features + ["HomeValue_to_Income"]
mortgage_features = common_features + [
    "HomeValue_to_Income",
    "HomeValue_Change",
    "MortgageValue_Change"
]

In [ ]:
merged["Rent_to_Mortgage_Ratio"] = merged["RentValue"] / merged["MortgageValue"]
merged["MortgageValue_Change"] = merged["MortgageValue"].pct_change()
merged["Sales_to_Inventory"] = merged["SalesCount"] / merged["Inventory"]

In [ ]:
merged.columns

Index(['RegionID', 'RegionName', 'StateName', 'Date', 'HomeValue',
       'IncomeNeeded', 'Inventory', 'DaysToPending', 'RentValue',
       'RenterIncomeNeeded', 'MarketHeatIndex', 'SalesCount',
       'NewConstruction', 'HomeValue_lag1', 'Inventory_lag1', 'RentValue_lag1',
       'MarketHeatIndex_lag1', 'HomeValue_roll3', 'Inventory_roll3',
       'RentValue_roll3', 'MarketHeatIndex_roll3', 'cpi', 'unrate',
       'mortgage_rate', 'Month', 'Month_Sin', 'Month_Cos',
       'HomeValue_to_Income', 'MortgageBurden', 'Inventory_to_Sales',
       'HomeValue_Change', 'IncomeNeeded_Change', 'CPI_Change',
       'MortgageRate_Change', 'Unemployment_Change', 'cpi_lag1', 'cpi_roll3',
       'mortgage_rate_lag1', 'mortgage_rate_roll3', 'unrate_lag1',
       'unrate_roll3', 'MortgageValue', 'Rent_to_Mortgage_Ratio',
       'MortgageValue_Change', 'Sales_to_Inventory'],
      dtype='object')

In [ ]:
merged["MortgageValue_Change"] = merged["MortgageValue_Change"].fillna(0)

In [ ]:
merged.to_csv("rent_mortgage_all_features.csv", index=False)

In [ ]:
def evaluate_metro(df, model, target, features, params=None, test_months=12):
  df = df.sort_values("Date").copy()
  train = df.iloc[:-test_months]
  test  = df.iloc[-test_months:]
  X_train, y_train = train[features], train[target]
  X_test,  y_test  = test[features],  test[target]
  evaluate_model = model(**params) if params else model()
  evaluate_model.fit(X_train, y_train)
  pred = evaluate_model.predict(X_test)
  mae  = mean_absolute_error(y_test, pred)
  rmse = np.sqrt(mean_squared_error(y_test, pred))
  return mae, rmse

In [ ]:
results = []
target = "RentValue"
for region, df_region in merged.groupby("RegionID"):
  if len(df_region) > 15:
    MODEL_CLASS = LinearRegression
    mae, rmse = evaluate_metro(df_region, MODEL_CLASS, target, rent_features)
    results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE": mae,
        "RMSE": rmse,
    })

results_df = pd.DataFrame(results)

In [ ]:
corrs = merged[rent_features + ["RentValue"]].corr(numeric_only=True)['RentValue'].sort_values(ascending=False)
print(corrs)

RentValue                 1.000000
MortgageBurden            0.421101
cpi                       0.403542
mortgage_rate             0.324944
SalesCount                0.288513
Inventory                 0.285717
MarketHeatIndex           0.164227
MortgageRate_Change       0.105926
NewConstruction           0.093180
CPI_Change                0.069511
Month_Sin                 0.024877
Sales_to_Inventory        0.018368
Month_Cos                 0.001093
Inventory_to_Sales       -0.018238
Unemployment_Change      -0.025208
HomeValue_to_Income      -0.061123
DaysToPending            -0.091362
unrate                   -0.124205
Rent_to_Mortgage_Ratio   -0.598393
Name: RentValue, dtype: float64


In [ ]:
results_df[['MAE','RMSE']].mean()

,0
MAE,25.885328
RMSE,28.281559


In [ ]:
results = []
target = "MortgageValue"

for region, df_region in merged.groupby("RegionID"):
  if len(df_region) > 15:
    MODEL_CLASS = LinearRegression
    mae, rmse = evaluate_metro(df_region, MODEL_CLASS, target, mortgage_features)
    results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE": mae,
        "RMSE": rmse,
    })

mortgage_df = pd.DataFrame(results)

In [ ]:
mortgage_df[['MAE','RMSE']].mean()

,0
MAE,47.152221
RMSE,51.679770


In [ ]:
corrs = merged[mortgage_features + ["MortgageValue"]].corr(numeric_only=True)['MortgageValue'].sort_values(ascending=False)
print(corrs)

MortgageValue             1.000000
HomeValue                 0.948453
MortgageBurden            0.589007
cpi                       0.485103
mortgage_rate             0.474982
MarketHeatIndex           0.138161
MortgageRate_Change       0.098312
MortgageValue_Change      0.066978
Inventory                 0.064013
SalesCount                0.061474
CPI_Change                0.048362
Month_Sin                 0.018644
NewConstruction           0.017280
Month_Cos                -0.005751
Sales_to_Inventory       -0.009035
Inventory_to_Sales       -0.021977
Unemployment_Change      -0.025822
DaysToPending            -0.124720
HomeValue_to_Income      -0.164307
unrate                   -0.194720
HomeValue_Change         -0.241256
Rent_to_Mortgage_Ratio   -0.771052
Name: MortgageValue, dtype: float64


In [ ]:
#Prophet
def evaluate_metro_prophet(df, target, regressors, test_months=12):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df = df.rename(columns={"Date": "ds", target: "y"}).dropna(subset=["y"])
    numeric_cols = [c for c in numeric_cols if c != target]
    df = df.set_index("ds").resample("ME")[numeric_cols + ["y"]].mean().reset_index()
    train = df.iloc[:-test_months]
    test  = df.iloc[-test_months:]

    model = Prophet(yearly_seasonality=True)
    for reg in regressors:
        model.add_regressor(reg)

    model.fit(train[["ds", "y"] + regressors])

    future = model.make_future_dataframe(periods=test_months, freq="ME")
    future = future.merge(df[["ds"] + regressors], on="ds", how="left")

    forecast = model.predict(future)

    y_pred = forecast["yhat"][-test_months:].values
    y_true = test["y"].values

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {"MAE": mae, "RMSE": rmse, "Forecast": forecast}

In [ ]:
prophet_results = []
rent_regressors = [
    "HomeValue",       # property value influences rent
    "Inventory",       # supply pressure
    "SalesCount",      # market activity
    "MarketHeatIndex", # local market trend
    "cpi",             # inflation
    "unrate",          # unemployment
    "mortgage_rate",   # mortgage rates can indirectly affect rent
    "MortgageBurden",  # helps capture cost-of-living influence
]


for region, df_region in merged.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    results = evaluate_metro_prophet(df_region, "RentValue", rent_regressors)
    prophet_results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE_prophet": results["MAE"],
        "RMSE_prophet": results["RMSE"],
        "Forecast": results["Forecast"]
    })

prophet_results_df = pd.DataFrame(prophet_results)

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmp0dtom1p2/b2sgv533.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp0dtom1p2/7jvxic75.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=15099', 'data', 'file=/tmp/tmp0dtom1p2/b2sgv533.json', 'init=/tmp/tmp0dtom1p2/7jvxic75.json', 'output', 'file=/tmp/tmp0dtom1p2/prophet_model1sffa8mk/prophet_model-20251026125828.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
12:58:28 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
12:58:29 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonali

In [ ]:
prophet_results_df[["MAE_prophet","RMSE_prophet"]].mean()

,0
MAE_prophet,15.239458
RMSE_prophet,17.207582


In [ ]:
prophet_results = []
mortgage_regressors = [
    "HomeValue",      # biggest driver of mortgage payment
    "MortgageBurden", # cost sensitivity to rates
    "cpi",            # macroeconomic factor
    "mortgage_rate",  # interest rate
    "SalesCount",     # optional market activity signal
    "Inventory",      # supply factor
    "MarketHeatIndex" # optional, trend indicator
]

for region, df_region in merged.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    results = evaluate_metro_prophet(df_region, "MortgageValue", mortgage_regressors)
    prophet_results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "MAE_prophet": results["MAE"],
        "RMSE_prophet": results["RMSE"],
        "Forecast": results["Forecast"]
    })

prophet_mortgage_results_df = pd.DataFrame(prophet_results)

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmp0dtom1p2/u1ec0u93.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmp0dtom1p2/p4bentyc.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.12/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=13379', 'data', 'file=/tmp/tmp0dtom1p2/u1ec0u93.json', 'init=/tmp/tmp0dtom1p2/p4bentyc.json', 'output', 'file=/tmp/tmp0dtom1p2/prophet_model47cuug0y/prophet_model-20251026130053.csv', 'method=optimize', 'algorithm=newton', 'iter=10000']
13:00:53 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
13:00:53 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing
INFO:prophet:Disabling weekly seasonali

In [ ]:
prophet_mortgage_results_df[["MAE_prophet","RMSE_prophet"]].mean()

,0
MAE_prophet,12.087762
RMSE_prophet,13.837960


In [ ]:
results_xgb = []
params = {
    "n_estimators": 200,
    "learning_rate": 0.1,
    "max_depth": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

for region, df_region in merged.groupby("RegionID"):
  df_region = df_region.sort_values("Date").dropna()
  if len(df_region) < 24:
    continue
  MODEL_CLASS =  XGBRegressor
  mae, rmse = evaluate_metro(df_region, MODEL_CLASS, "RentValue", rent_features, params)
  results_xgb.append({
    "RegionID": region,
    "RegionName": df_region["RegionName"].iloc[0],
    "StateName": df_region["StateName"].iloc[0],
    "MAE_xgb": mae,
    "RMSE_xgb": rmse,
  })

results_xgb_df = pd.DataFrame(results_xgb)

In [ ]:
results_xgb_df[["MAE_xgb","RMSE_xgb"]].mean()

,0
MAE_xgb,41.764160
RMSE_xgb,44.650438


In [ ]:
results_xgb = []
params = {
    "n_estimators": 200,
    "learning_rate": 0.1,
    "max_depth": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

for region, df_region in merged.groupby("RegionID"):
  df_region = df_region.sort_values("Date").dropna()
  if len(df_region) < 24:
    continue
  MODEL_CLASS =  XGBRegressor
  mae, rmse = evaluate_metro(df_region, MODEL_CLASS, "MortgageValue", mortgage_features, params)
  results_xgb.append({
    "RegionID": region,
    "RegionName": df_region["RegionName"].iloc[0],
    "StateName": df_region["StateName"].iloc[0],
    "MAE_xgb": mae,
    "RMSE_xgb": rmse,
  })

results_mortgage_xgb_df = pd.DataFrame(results_xgb)

In [ ]:
results_mortgage_xgb_df[["MAE_xgb","RMSE_xgb"]].mean()

,0
MAE_xgb,66.686093
RMSE_xgb,71.945572


In [ ]:
def prepare_lstm_data(df, feature_cols, target_col, lookback=12):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[feature_cols + [target_col]])
    X, y = [], []
    for i in range(lookback, len(scaled)):
        X.append(scaled[i-lookback:i, :-1])
        y.append(scaled[i, -1])
    X, y = np.array(X), np.array(y)
    return X, y, scaler

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

In [ ]:
def evaluate_lstm_all_metros(df, feature_cols, target_col, test_months=12,
                             lookback=12, epochs=20, batch_size=8):
    results = []

    for metro in df["RegionName"].unique():
        df_metro = df[df["RegionName"] == metro].sort_values("Date").copy()

        if len(df_metro) < (lookback + test_months + 1):
            continue

        # Prepare data
        X, y, scaler = prepare_lstm_data(df_metro, feature_cols, target_col, lookback)

        # Train/test split
        X_train, X_test = X[:-test_months], X[-test_months:]
        y_train, y_test = y[:-test_months], y[-test_months:]

        # Build model
        model = build_lstm_model((X.shape[1], X.shape[2]))

        # Fit model
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

        # Predict
        y_pred = model.predict(X_test, verbose=0)

        # Reconstruct arrays for inverse scaling
        y_test_full = np.hstack([np.zeros((len(y_test), len(feature_cols))), y_test.reshape(-1,1)])
        y_pred_full = np.hstack([np.zeros((len(y_pred), len(feature_cols))), y_pred])

        # Inverse scale
        y_test_rescaled = scaler.inverse_transform(y_test_full)[:, -1]
        y_pred_rescaled = scaler.inverse_transform(y_pred_full)[:, -1]

        # Metrics
        mae = mean_absolute_error(y_test_rescaled, y_pred_rescaled)
        rmse = np.sqrt(mean_squared_error(y_test_rescaled, y_pred_rescaled))

        results.append({
            "RegionName": metro,
            "LSTM_MAE": mae,
            "LSTM_RMSE": rmse
        })

    return pd.DataFrame(results)

In [ ]:
lstm_rent_features = [
    "HomeValue", "Inventory", "SalesCount", "MarketHeatIndex",
    "NewConstruction", "cpi", "unrate", "mortgage_rate",
    "Month_Sin", "Month_Cos",
    "MortgageBurden", "HomeValue_to_Income",
    "HomeValue_lag1", "Inventory_lag1", "MarketHeatIndex_lag1",
    "HomeValue_roll3", "Inventory_roll3", "MarketHeatIndex_roll3",
    "RentValue_lag1", "RentValue_roll3"
]

lstm_results = evaluate_lstm_all_metros(
    df=merged,
    feature_cols=lstm_rent_features,
    target_col="RentValue",
    test_months=12,
    lookback=12,
    epochs=20,
    batch_size=8
)

In [ ]:

lstm_results[["LSTM_MAE","LSTM_RMSE"]].mean()

,0
LSTM_MAE,47.900120
LSTM_RMSE,50.506871


In [ ]:
lstm_mortgage_features = [
    "HomeValue", "Inventory", "SalesCount", "MarketHeatIndex",
    "NewConstruction", "cpi", "unrate", "mortgage_rate",
    "Month_Sin", "Month_Cos",
    "MortgageBurden", "MortgageRate_Change", "CPI_Change",
    "HomeValue_lag1", "Inventory_lag1", "MarketHeatIndex_lag1",
    "HomeValue_roll3", "Inventory_roll3", "MarketHeatIndex_roll3",
    "mortgage_rate_lag1", "mortgage_rate_roll3",
]

lstm_mortgage_results = evaluate_lstm_all_metros(
    df=merged,
    feature_cols=lstm_mortgage_features,
    target_col="MortgageValue",
    test_months=12,
    lookback=12,
    epochs=20,
    batch_size=8
)

In [ ]:
lstm_mortgage_results[["LSTM_MAE","LSTM_RMSE"]].mean()

,0
LSTM_MAE,76.973874
LSTM_RMSE,93.631550


In [ ]:
print("Error Ranking Table for Rent Validation")
rent_error_table = pd.DataFrame({
    "MAE_mean": [
        results_df["MAE"].mean(),
        results_xgb_df["MAE_xgb"].mean(),
        prophet_results_df["MAE_prophet"].mean(),
        lstm_results["LSTM_MAE"].mean()
    ],
    "RMSE_mean": [
        results_df["RMSE"].mean(),
        results_xgb_df["RMSE_xgb"].mean(),
        prophet_results_df["RMSE_prophet"].mean(),
        lstm_results["LSTM_RMSE"].mean()
    ]
}, index=["Linear Regression", "XGBoost", "Prophet", "LSTM"]).sort_values("MAE_mean")
rent_error_table

Error Ranking Table for Rent Validation


,MAE_mean,RMSE_mean
Prophet,15.239458,17.207582
Linear Regression,25.885328,28.281559
XGBoost,41.764160,44.650438
LSTM,47.900120,50.506871


In [ ]:
print("Error Ranking Table for Mortgage Validation")
rent_error_table = pd.DataFrame({
    "MAE_mean": [
        mortgage_df["MAE"].mean(),
        results_mortgage_xgb_df["MAE_xgb"].mean(),
        prophet_mortgage_results_df["MAE_prophet"].mean(),
        lstm_mortgage_results["LSTM_MAE"].mean()
    ],
    "RMSE_mean": [
        mortgage_df["RMSE"].mean(),
        results_mortgage_xgb_df["RMSE_xgb"].mean(),
        prophet_mortgage_results_df["RMSE_prophet"].mean(),
        lstm_mortgage_results["LSTM_RMSE"].mean()
    ]
}, index=["Linear Regression", "XGBoost", "Prophet", "LSTM"]).sort_values("MAE_mean")
rent_error_table

Error Ranking Table for Mortgage Validation


,MAE_mean,RMSE_mean
Prophet,12.087762,13.837960
Linear Regression,47.152221,51.679770
XGBoost,66.686093,71.945572
LSTM,76.973874,93.631550
